# Финальные эксперименты: Seed Averaging + Tweedie + Pseudo-Labeling

## 4 этапа:
1. **Seed Averaging** — обучение v4a с 5 seeds, усреднение → снижение дисперсии
2. **Tweedie Loss** — специализированная функция потерь для zero-inflated данных
3. **Pseudo-Labeling** — дообучение на тесте, размеченном лучшим блендом
4. **Финальные бленды** — микро-тюнинг весов для сабмитов

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
import time

FEATURES_DIR = Path("../data/processed/features_v4")
MODELS_DIR = Path("../models")
PROCESSED = Path("../data/processed")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 2

def rmsle_score(y_true, y_pred):
    y_pred_clipped = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_clipped)))

def gini_normalized(y_true, y_pred):
    def _gini(actual, predicted):
        n = len(actual)
        indices = np.argsort(-predicted)
        sorted_actual = actual[indices]
        cumulative = np.cumsum(sorted_actual)
        gini_sum = cumulative.sum() / (sorted_actual.sum() + 1e-9) - (n + 1) / 2
        return gini_sum / n
    return _gini(y_true, y_pred) / (_gini(y_true, y_true) + 1e-9)

def load_fold(fold_path: Path) -> pd.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").to_pandas()

print("Загрузка данных...")
all_folds = []
for fold_idx in range(N_FOLDS):
    fold_df = load_fold(FEATURES_DIR / f"fold_{fold_idx:02d}")
    all_folds.append(fold_df)
    n_buyers = (fold_df["target"] > 0).sum()
    print(f"  fold_{fold_idx:02d}: {len(fold_df):,} | buyers={n_buyers:,} ({n_buyers/len(fold_df)*100:.1f}%)")

test_df = load_fold(FEATURES_DIR / "fold_test")
print(f"  fold_test: {len(test_df):,}")

drop_cols = ["user_id", "anchor_date", "target"]
features = [c for c in test_df.columns if c not in drop_cols]
print(f"Фичей: {len(features)}")

Загрузка данных...
  fold_00: 250,000 | buyers=133,886 (53.6%)
  fold_01: 250,000 | buyers=135,165 (54.1%)
  fold_test: 250,000
Фичей: 392


---
## Этап 1: Seed Averaging (v4a × 5 seeds)

Обучаем CatBoost v4a (baseline params: iter=2000, lr=0.03, depth=6) с 5 разными seeds.
Усреднение предсказаний снижает дисперсию без увеличения bias.

In [2]:
SEEDS = [42, 123, 777, 2024, 31415]

baseline_params_template = {
    'iterations': 2000,
    'learning_rate': 0.03,
    'depth': 6,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'task_type': 'GPU',
    'devices': '0',
    'od_type': 'Iter',
    'early_stopping_rounds': 150,
    'verbose': 500,
}

train_df = all_folds[0]
val_df = all_folds[1]

X_train = train_df[features]
y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
X_val = val_df[features]
y_val = val_df["target"].values
y_val_log = np.log1p(np.clip(y_val, 0, None))

train_pool = Pool(X_train, y_train_log)
val_pool = Pool(X_val, y_val_log)

seed_test_preds = []
seed_val_preds = []
seed_scores = []

print("=== Этап 1: Seed Averaging ===")
t0 = time.time()

for i, seed in enumerate(SEEDS):
    params = {**baseline_params_template, 'random_seed': seed}
    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool)
    
    val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
    test_pred = np.expm1(np.clip(model.predict(test_df[features]), 0, None))
    
    score = rmsle_score(y_val, val_pred)
    seed_scores.append(score)
    seed_val_preds.append(val_pred)
    seed_test_preds.append(test_pred)
    
    print(f"  Seed {seed}: RMSLE={score:.5f} ({time.time()-t0:.0f}s)")

avg_val_pred = np.mean(seed_val_preds, axis=0)
avg_test_pred = np.mean(seed_test_preds, axis=0)

avg_score = rmsle_score(y_val, avg_val_pred)
avg_gini = gini_normalized(y_val, avg_val_pred)

print(f"\n  Seed Avg RMSLE: {avg_score:.5f} (vs seed42: {seed_scores[0]:.5f}, Δ={avg_score-seed_scores[0]:+.5f})")
print(f"  Seed Avg Gini:  {avg_gini:.4f}")
print(f"  Индив. scores:  {[f'{s:.5f}' for s in seed_scores]}")
print(f"  Std seeds:      {np.std(seed_scores):.5f}")

sub_seed = test_df[["user_id"]].copy()
sub_seed["predict"] = np.clip(avg_test_pred, 0, None)
sub_seed.to_csv(PROCESSED / "catboost_v4a_seed_avg.csv", index=False)
print(f"\nСохранено: catboost_v4a_seed_avg.csv (mean={sub_seed['predict'].mean():.2f})")

=== Этап 1: Seed Averaging ===
0:	learn: 2.2638575	test: 2.2596438	best: 2.2596438 (0)	total: 129ms	remaining: 4m 17s
500:	learn: 1.6774814	test: 1.6693647	best: 1.6693647 (500)	total: 5.93s	remaining: 17.8s
1000:	learn: 1.6646071	test: 1.6665827	best: 1.6665827 (1000)	total: 11.4s	remaining: 11.3s
1500:	learn: 1.6532232	test: 1.6647400	best: 1.6647400 (1500)	total: 16.7s	remaining: 5.55s
1999:	learn: 1.6424919	test: 1.6633227	best: 1.6633227 (1999)	total: 22.3s	remaining: 0us
bestTest = 1.663322729
bestIteration = 1999
  Seed 42: RMSLE=1.66332 (25s)
0:	learn: 2.2637901	test: 2.2598340	best: 2.2598340 (0)	total: 18.9ms	remaining: 37.7s
500:	learn: 1.6773429	test: 1.6694468	best: 1.6694468 (500)	total: 6.16s	remaining: 18.4s
1000:	learn: 1.6643392	test: 1.6666043	best: 1.6666043 (1000)	total: 11.5s	remaining: 11.5s
1500:	learn: 1.6529859	test: 1.6648905	best: 1.6648905 (1500)	total: 16.8s	remaining: 5.57s
1999:	learn: 1.6424250	test: 1.6636000	best: 1.6636000 (1999)	total: 22.2s	remaini

---
## Этап 2: Tweedie Loss

Tweedie distribution (power=1.5) — специально для zero-inflated данных с тяжёлым хвостом.
В отличие от RMSE на log1p, Tweedie напрямую моделирует zero-inflated распределение.

**Важно:** Tweedie работает на ИСХОДНОМ таргете (без log1p!)

In [3]:
print("=== Этап 2: Tweedie Loss ===")

tweedie_test_preds = []
tweedie_val_preds = []
tweedie_scores = []

y_train_raw = np.clip(train_df["target"].values, 0, None)
y_val_raw = np.clip(val_df["target"].values, 0, None)

train_pool_raw = Pool(X_train, y_train_raw)
val_pool_raw = Pool(X_val, y_val_raw)

for power in [1.3, 1.5, 1.7, 1.9]:
    tweedie_params = {
        'iterations': 2000,
        'learning_rate': 0.03,
        'depth': 6,
        'loss_function': f'Tweedie:variance_power={power}',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 42,
        'od_type': 'Iter',
        'early_stopping_rounds': 150,
        'verbose': 500,
    }
    
    model_tw = CatBoostRegressor(**tweedie_params)
    model_tw.fit(train_pool_raw, eval_set=val_pool_raw)

    val_pred_tw = np.clip(model_tw.predict(X_val), 0, None)
    test_pred_tw = np.clip(model_tw.predict(test_df[features]), 0, None)
    
    score_tw = rmsle_score(y_val, val_pred_tw)
    gini_tw = gini_normalized(y_val, val_pred_tw)
    tweedie_scores.append((power, score_tw, gini_tw))
    
    print(f"  Tweedie(p={power}): RMSLE={score_tw:.5f} | Gini={gini_tw:.4f} | mean={val_pred_tw.mean():.2f}")
    
    tweedie_val_preds.append(val_pred_tw)
    tweedie_test_preds.append(test_pred_tw)

best_tw_idx = np.argmin([s[1] for s in tweedie_scores])
best_tw_power, best_tw_score, best_tw_gini = tweedie_scores[best_tw_idx]
best_tw_test = tweedie_test_preds[best_tw_idx]
best_tw_val = tweedie_val_preds[best_tw_idx]

print(f"\n  Лучший Tweedie: power={best_tw_power}, RMSLE={best_tw_score:.5f}")
print(f"  vs RMSE+log1p (seed avg): {avg_score:.5f}")

sub_tw = test_df[["user_id"]].copy()
sub_tw["predict"] = np.clip(best_tw_test, 0, None)
sub_tw.to_csv(PROCESSED / f"catboost_v4a_tweedie_p{best_tw_power}.csv", index=False)
print(f"Сохранено: catboost_v4a_tweedie_p{best_tw_power}.csv")

print(f"\n--- Seed Averaging лучшего Tweedie (power={best_tw_power}) ---")
tw_seed_test_preds = []
tw_seed_val_preds = []

for seed in SEEDS:
    tw_params_s = {
        'iterations': 2000,
        'learning_rate': 0.03,
        'depth': 6,
        'loss_function': f'Tweedie:variance_power={best_tw_power}',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': seed,
        'od_type': 'Iter',
        'early_stopping_rounds': 150,
        'verbose': False,
    }
    m = CatBoostRegressor(**tw_params_s)
    m.fit(train_pool_raw, eval_set=val_pool_raw)
    
    tw_seed_val_preds.append(np.clip(m.predict(X_val), 0, None))
    tw_seed_test_preds.append(np.clip(m.predict(test_df[features]), 0, None))
    print(f"  Seed {seed} done")

tw_avg_val = np.mean(tw_seed_val_preds, axis=0)
tw_avg_test = np.mean(tw_seed_test_preds, axis=0)
tw_avg_score = rmsle_score(y_val, tw_avg_val)
print(f"  Tweedie Seed Avg RMSLE: {tw_avg_score:.5f}")

sub_tw_avg = test_df[["user_id"]].copy()
sub_tw_avg["predict"] = tw_avg_test
sub_tw_avg.to_csv(PROCESSED / f"catboost_v4a_tweedie_seed_avg.csv", index=False)
print(f"Сохранено: catboost_v4a_tweedie_seed_avg.csv")

=== Этап 2: Tweedie Loss ===
0:	learn: 297.8537172	test: 291.8084474	best: 291.8084474 (0)	total: 16.6ms	remaining: 33.3s
bestTest = 291.8084474
bestIteration = 0
Shrink model to first 1 iterations.
  Tweedie(p=1.3): RMSLE=2.76325 | Gini=-0.0047 | mean=1.00
0:	learn: 297.8537172	test: 291.8084474	best: 291.8084474 (0)	total: 14.7ms	remaining: 29.4s
bestTest = 291.8084474
bestIteration = 0
Shrink model to first 1 iterations.
  Tweedie(p=1.5): RMSLE=2.76325 | Gini=-0.0047 | mean=1.00
0:	learn: 297.8537172	test: 291.8084474	best: 291.8084474 (0)	total: 12.4ms	remaining: 24.9s
bestTest = 291.8084474
bestIteration = 0
Shrink model to first 1 iterations.
  Tweedie(p=1.7): RMSLE=2.76325 | Gini=-0.0047 | mean=1.00
0:	learn: 297.8537172	test: 291.8084474	best: 291.8084474 (0)	total: 14.8ms	remaining: 29.6s
bestTest = 291.8084474
bestIteration = 0
Shrink model to first 1 iterations.
  Tweedie(p=1.9): RMSLE=2.76325 | Gini=-0.0047 | mean=1.00

  Лучший Tweedie: power=1.3, RMSLE=2.76325
  vs RMSE+l

---
## Этап 3: Pseudo-Labeling

Дообучение CatBoost на тестовой выборке, размеченной текущим лучшим блендом.

In [4]:
print("=== Этап 3: Pseudo-Labeling ===")

best_blend_path = PROCESSED / "blend_ortho_90_10.csv"
if best_blend_path.exists():
    best_blend = pd.read_csv(best_blend_path).sort_values("user_id").reset_index(drop=True)
    pseudo_labels = best_blend["predict"].values
    print(f"  Pseudo-labels загружены: mean={pseudo_labels.mean():.2f}, median={np.median(pseudo_labels):.2f}")
else:
    pseudo_labels = avg_test_pred
    print(f"  Используем seed_avg как pseudo-labels: mean={pseudo_labels.mean():.2f}")

X_test_features = test_df[features]
y_pseudo_log = np.log1p(np.clip(pseudo_labels, 0, None))

X_combined = pd.concat([X_train, X_test_features], ignore_index=True)
y_combined = np.concatenate([y_train_log, y_pseudo_log])

print(f"  Train: {len(X_train):,} + Pseudo: {len(X_test_features):,} = {len(X_combined):,}")

pl_test_preds = []
pl_val_preds = []

for seed in SEEDS:
    pl_params = {
        **baseline_params_template,
        'random_seed': seed,
        'verbose': False,
    }
    
    combined_pool = Pool(X_combined, y_combined)
    
    model_pl = CatBoostRegressor(**pl_params)
    model_pl.fit(combined_pool, eval_set=val_pool)
    
    val_pred_pl = np.expm1(np.clip(model_pl.predict(X_val), 0, None))
    test_pred_pl = np.expm1(np.clip(model_pl.predict(X_test_features), 0, None))
    
    pl_val_preds.append(val_pred_pl)
    pl_test_preds.append(test_pred_pl)
    print(f"  Seed {seed}: val RMSLE={rmsle_score(y_val, val_pred_pl):.5f}")

pl_avg_val = np.mean(pl_val_preds, axis=0)
pl_avg_test = np.mean(pl_test_preds, axis=0)
pl_score = rmsle_score(y_val, pl_avg_val)
print(f"\n  Pseudo-Label Seed Avg RMSLE: {pl_score:.5f}")
print(f"  vs Normal Seed Avg:          {avg_score:.5f}")

sub_pl = test_df[["user_id"]].copy()
sub_pl["predict"] = np.clip(pl_avg_test, 0, None)
sub_pl.to_csv(PROCESSED / "catboost_v4a_pseudo_label.csv", index=False)
print(f"Сохранено: catboost_v4a_pseudo_label.csv")

=== Этап 3: Pseudo-Labeling ===
  Pseudo-labels загружены: mean=35.76, median=6.61
  Train: 250,000 + Pseudo: 250,000 = 500,000
  Seed 42: val RMSLE=1.66621
  Seed 123: val RMSLE=1.66607
  Seed 777: val RMSLE=1.66631
  Seed 2024: val RMSLE=1.66616
  Seed 31415: val RMSLE=1.66612

  Pseudo-Label Seed Avg RMSLE: 1.66596
  vs Normal Seed Avg:          1.66297
Сохранено: catboost_v4a_pseudo_label.csv


---
## Этап 4: Финальные бленды

Микро-тюнинг весов для всех комбинаций.

In [5]:
print("=== Этап 4: Финальные бленды ===")

rnn_v2 = pd.read_csv(PROCESSED / "rnn_v2_submission.csv").sort_values("user_id").reset_index(drop=True)
p_rnn = rnn_v2["predict"].values

cumul_path = PROCESSED / "catboost_v4_cumul_submission.csv"
if cumul_path.exists():
    v4_cumul_df = pd.read_csv(cumul_path).sort_values("user_id").reset_index(drop=True)
    p_cumul = v4_cumul_df["predict"].values
    has_cumul = True
    print(f"  v4a_cumul loaded: mean={p_cumul.mean():.2f}")
else:
    has_cumul = False
    print("  v4a_cumul не найден")

test_sorted = test_df[["user_id"]].copy().sort_values("user_id").reset_index(drop=True)
user_ids = test_sorted["user_id"].values

models = {}

seed_df = pd.DataFrame({"user_id": test_df["user_id"], "predict": avg_test_pred}).sort_values("user_id").reset_index(drop=True)
models["seed_avg"] = seed_df["predict"].values

tw_df = pd.DataFrame({"user_id": test_df["user_id"], "predict": tw_avg_test}).sort_values("user_id").reset_index(drop=True)
models["tweedie_avg"] = tw_df["predict"].values

pl_df = pd.DataFrame({"user_id": test_df["user_id"], "predict": pl_avg_test}).sort_values("user_id").reset_index(drop=True)
models["pseudo_label"] = pl_df["predict"].values

v4a_df = pd.read_csv(PROCESSED / "catboost_v4a_baseline_submission.csv").sort_values("user_id").reset_index(drop=True)
models["v4a"] = v4a_df["predict"].values

models["rnn_v2"] = p_rnn

print(f"\n  Модели для блендинга: {list(models.keys())}")
for name, preds in models.items():
    print(f"    {name}: mean={preds.mean():.2f}, median={np.median(preds):.2f}")

submissions = {}

for rnn_w in [0.05, 0.08, 0.10, 0.12, 0.15]:
    cb_w = 1.0 - rnn_w
    blend = cb_w * models["seed_avg"] + rnn_w * models["rnn_v2"]
    name = f"blend_seed_avg_{int(cb_w*100)}_rnn{int(rnn_w*100)}"
    submissions[name] = blend

for rnn_w in [0.05, 0.10, 0.15]:
    cb_w = 1.0 - rnn_w
    blend = cb_w * models["tweedie_avg"] + rnn_w * models["rnn_v2"]
    name = f"blend_tweedie_{int(cb_w*100)}_rnn{int(rnn_w*100)}"
    submissions[name] = blend

for rnn_w in [0.05, 0.10]:
    cb_w = 1.0 - rnn_w
    blend = cb_w * models["pseudo_label"] + rnn_w * models["rnn_v2"]
    name = f"blend_pseudo_{int(cb_w*100)}_rnn{int(rnn_w*100)}"
    submissions[name] = blend

blend_3m = 0.45 * models["seed_avg"] + 0.45 * models["tweedie_avg"] + 0.10 * models["rnn_v2"]
submissions["blend_3model_sa45_tw45_rnn10"] = blend_3m

if has_cumul:
    blend_cum = 0.85 * models["seed_avg"] + 0.05 * p_cumul + 0.10 * models["rnn_v2"]
    submissions["blend_sa85_cum5_rnn10"] = blend_cum

print(f"\n--- Все бленды ---")
for name, blend in submissions.items():
    blend_clipped = np.clip(blend, 0, None)
    pd.DataFrame({"user_id": user_ids, "predict": blend_clipped}).to_csv(
        PROCESSED / f"{name}.csv", index=False)
    print(f"  {name}: mean={blend_clipped.mean():.2f}")

=== Этап 4: Финальные бленды ===
  v4a_cumul не найден

  Модели для блендинга: ['seed_avg', 'tweedie_avg', 'pseudo_label', 'v4a', 'rnn_v2']
    seed_avg: mean=35.34, median=6.20
    tweedie_avg: mean=1.00, median=1.00
    pseudo_label: mean=35.49, median=6.41
    v4a: mean=35.35, median=6.18
    rnn_v2: mean=39.45, median=8.44

--- Все бленды ---
  blend_seed_avg_95_rnn5: mean=35.54
  blend_seed_avg_92_rnn8: mean=35.67
  blend_seed_avg_90_rnn10: mean=35.75
  blend_seed_avg_88_rnn12: mean=35.83
  blend_seed_avg_85_rnn15: mean=35.95
  blend_tweedie_95_rnn5: mean=2.92
  blend_tweedie_90_rnn10: mean=4.84
  blend_tweedie_85_rnn15: mean=6.77
  blend_pseudo_95_rnn5: mean=35.69
  blend_pseudo_90_rnn10: mean=35.89
  blend_3model_sa45_tw45_rnn10: mean=20.30


---
## Итоговая сводка

In [ ]:
print("="*60)
print("ИТОГОВАЯ СВОДКА")
print("="*60)
print(f"\nCV метрики (на fold_01):")
print(f"  v4a seed=42:       {seed_scores[0]:.5f}")
print(f"  v4a Seed Avg:      {avg_score:.5f} (Δ={avg_score-seed_scores[0]:+.5f})")
print(f"  Tweedie best:      {best_tw_score:.5f} (p={best_tw_power})")
print(f"  Tweedie Seed Avg:  {tw_avg_score:.5f}")
print(f"  Pseudo-Label Avg:  {pl_score:.5f}")

print(f"\nРекомендуемые сабмиты (топ-5):")
print(f"  1. blend_seed_avg_90_rnn10.csv  — основной, seed avg снижает дисперсию")
print(f"  2. blend_seed_avg_92_rnn8.csv   — чуть больше CB")
print(f"  3. blend_tweedie_90_rnn10.csv   — Tweedie если CV хороший")
print(f"  4. blend_3model_sa45_tw45_rnn10.csv — диверсификация")
print(f"  5. blend_pseudo_90_rnn10.csv    — pseudo-labeling")

ИТОГОВАЯ СВОДКА

CV метрики (на fold_01):
  v4a seed=42:       1.66332
  v4a Seed Avg:      1.66297 (Δ=-0.00035)
  Tweedie best:      2.76325 (p=1.3)
  Tweedie Seed Avg:  2.76325
  Pseudo-Label Avg:  1.66596

Рекомендуемые сабмиты (топ-5):
  1. blend_seed_avg_90_rnn10.csv  — основной, seed avg снижает дисперсию
  2. blend_seed_avg_92_rnn8.csv   — чуть больше CB
  3. blend_tweedie_90_rnn10.csv   — Tweedie если CV хороший
  4. blend_3model_sa45_tw45_rnn10.csv — диверсификация
  5. blend_pseudo_90_rnn10.csv    — pseudo-labeling
